# Phase 2: OpenWeather Historical API Investigation

## 🎯 Objective
Verify the capabilities, boundaries, schemas, and limitations of the OpenWeather API before constructing data ingestion and feature engineering pipelines.

### Key Areas of Investigation:
1. **Historical Air Pollution Endpoint** (`/data/2.5/air_pollution/history`)
2. **Data Span & Temporal Resolution** (Nov 27, 2020 to present; 1-hour resolution)
3. **Pollutants & Schema Validation** ($CO, NO, NO_2, O_3, SO_2, PM_{2.5}, PM_{10}, NH_3$ in $\mu g/m^3$)
4. **AQI Standard Discrepancy** (OpenWeather European CAQI 1-5 vs Target US EPA AQI 0-500)
5. **Free Tier Rate Limits & Weather API Constraints**

In [ ]:
import os
import sys
from datetime import datetime, timezone
import requests
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Load environment variables
load_dotenv("../.env")

API_KEY = os.getenv("OPENWEATHER_API_KEY")
CITY_NAME = os.getenv("TARGET_CITY_NAME", "Lahore")
LAT = float(os.getenv("TARGET_LAT", "31.5497"))
LON = float(os.getenv("TARGET_LON", "74.3436"))

print(f"Target City: {CITY_NAME} (Lat: {LAT}, Lon: {LON})")
print(f"API Key configured: {'Yes' if API_KEY and API_KEY != 'your_openweather_api_key_here' else 'No'}")

## 1. Test Current Air Pollution Endpoint (`/data/2.5/air_pollution`)

In [ ]:
url_current_aq = f"https://api.openweathermap.org/data/2.5/air_pollution?lat={LAT}&lon={LON}&appid={API_KEY}"
resp_current = requests.get(url_current_aq)
print(f"Status: {resp_current.status_code}")

if resp_current.status_code == 200:
    data_current = resp_current.json()
    print("\nResponse JSON:")
    print(data_current)
    
    # Extract components
    comp = data_current['list'][0]['components']
    aqi_owm = data_current['list'][0]['main']['aqi']
    ts = datetime.fromtimestamp(data_current['list'][0]['dt'], tz=timezone.utc)
    
    print(f"\nTimestamp (UTC): {ts}")
    print(f"OpenWeather AQI (1-5 scale): {aqi_owm}")
    print("Pollutant concentrations (μg/m³):")
    for pollutant, val in comp.items():
        print(f"  - {pollutant.upper():<6}: {val:.2f} μg/m³")

## 2. Test Historical Air Pollution Endpoint (`/data/2.5/air_pollution/history`)

Testing boundary: November 27, 2020 (`1606482000` UTC) to November 30, 2020 (`1606741200` UTC).

In [ ]:
start_ts = 1606482000  # Nov 27, 2020 13:00:00 UTC
end_ts = 1606741200    # Nov 30, 2020 13:00:00 UTC

url_hist = f"https://api.openweathermap.org/data/2.5/air_pollution/history?lat={LAT}&lon={LON}&start={start_ts}&end={end_ts}&appid={API_KEY}"
resp_hist = requests.get(url_hist)
print(f"Status: {resp_hist.status_code}")

if resp_hist.status_code == 200:
    data_hist = resp_hist.json()
    records = data_hist.get("list", [])
    print(f"Retrieved {len(records)} hourly records for 72-hour window.")
    
    # Convert to DataFrame
    df_sample = pd.DataFrame([
        {
            "datetime_utc": datetime.fromtimestamp(item["dt"], tz=timezone.utc),
            "owm_caqi": item["main"]["aqi"],
            **item["components"]
        }
        for item in records
    ])
    
    print("\nSample Dataframe:")
    display(df_sample.head())

## 3. Verify AQI Scale Gap: CAQI (1-5) vs EPA AQI (0-500)

OpenWeather reports an index `main.aqi` between 1 and 5 (based on European CAQI index).
Our forecasting system predicts the **US EPA AQI (0-500 continuous scale)**.

EPA AQI formula:
$$I_p = \frac{I_{Hi} - I_{Lo}}{BP_{Hi} - BP_{Lo}} (C_p - BP_{Lo}) + I_{Lo}$$

Overall EPA AQI = $\max(I_{PM2.5}, I_{PM10}, I_{O_3}, I_{NO_2}, I_{SO_2}, I_{CO})$.

In [ ]:
# EPA PM2.5 Breakpoints
EPA_PM25_BREAKPOINTS = [
    (0.0, 9.0, 0, 50),         # Good
    (9.1, 35.4, 51, 100),      # Moderate
    (35.5, 55.4, 101, 150),    # Unhealthy for Sensitive Groups
    (55.5, 125.4, 151, 200),   # Unhealthy
    (125.5, 225.4, 201, 300),  # Very Unhealthy
    (225.5, 325.4, 301, 400),  # Hazardous
    (325.5, 500.4, 401, 500),  # Hazardous
]

def calculate_pm25_aqi(pm25_conc: float) -> int:
    for c_low, c_high, i_low, i_high in EPA_PM25_BREAKPOINTS:
        if c_low <= pm25_conc <= c_high:
            return round(((i_high - i_low) / (c_high - c_low)) * (pm25_conc - c_low) + i_low)
    if pm25_conc > 500.4:
        return 500
    return 0

if 'df_sample' in locals():
    df_sample['epa_aqi_pm25'] = df_sample['pm2_5'].apply(calculate_pm25_aqi)
    print("Comparison of OpenWeather CAQI (1-5) vs Computed US EPA AQI (0-500):")
    display(df_sample[['datetime_utc', 'pm2_5', 'owm_caqi', 'epa_aqi_pm25']].head(10))

## 4. Key Findings & Architectural Conclusions

| Parameter | OpenWeather Capability | Project Impact |
| :--- | :--- | :--- |
| **Historical Air Pollution** | Available from **Nov 27, 2020** to present | ✅ Full 5.7+ years (~50k hours) available on free tier |
| **Temporal Resolution** | Hourly timestamps (`dt` field in UTC) | ✅ Perfect for 72-hour forecasting |
| **Pollutant Types** | $CO, NO, NO_2, O_3, SO_2, PM_{2.5}, PM_{10}, NH_3$ in $\mu g/m^3$ | ✅ All critical criteria pollutants present |
| **Historical Weather Data** | Paywalled under One Call API 3.0 | ✅ Locked ADR: Train on pollutants-only for Phases 1-15 |
| **Rate Limit** | 60 calls/min, 1,000,000 calls/month | ✅ Monthly batch backfill will take ~70 calls (~2 minutes total) |
| **AQI Scale** | Returns 1-5 European CAQI integer | ✅ Must calculate EPA AQI (0-500) via piecewise linear formulas |